## Analysis Overview

### Motivation

CTCV J2056-3014 (hereafter CJ2056) is a cataclysmic variable with a 29.6 s white-dwarf spin period. A previous analysis (notebook `07`) used $\dot{P}$ alone to compare an intermediate polar (IP) model against a non-magnetic CV (nmCV) model. That analysis relied on **informative mass priors derived from X-ray spectral fits** (Salcedo+2024), which introduced a correlation between $M_{\rm WD}$ and $\dot{M}$ before the inference even began.

- Because the spectral-fit mass priors already encoded information from the same $L_X$ and $kT$ data we want to use as constraints, using those priors here would amount to double-counting. We therefore replace them with **broad, population-based priors** and let the spectral observables enter through the likelihood instead.
- This notebook extends the model to **three observables** ($\dot{P}$, $kT_{\rm max}$, $L_X$), which jointly constrain the WD mass, accretion rate, and (for the IP) the magnetic dipole moment.

### Goal

Perform a Bayesian model comparison between two accretion scenarios for CJ2056:

- **nmCV (non-magnetic CV):** Disk extends to WD surface. 2 free parameters.
- **IP (intermediate polar):** Disk truncated at the magnetospheric radius $r_m$. 4 free parameters.

We quantify the relative evidence for each model via the Bayes factor, constrain physical parameters under each model, and assess what the new precise $\dot{P}$ measurement adds beyond spectral data.

---

### Observational Data

All measurements from Salcedo+2024 unless otherwise noted.

| Observable | Value | Uncertainty ($1\sigma$) | Source / Notes |
|:---|:---|:---|:---|
| $P$ (spin period) | 29.609685843 s | — | High-precision timing |
| $\dot{P}_{\rm obs}$ | 0.0 s/s | $1.7 \times 10^{-15}$ s/s | Salcedo+2024 Section 3.3; null detection |
| $d$ (distance) | 261.6 pc | 7.4 pc | Gaia parallax |
| $F_X$ (unabsorbed flux, 0.3–12 keV) | $2.2 \times 10^{-12}$ erg s$^{-1}$ cm$^{-2}$ | 5% (spectral fit stat. error) | Salcedo+2024 |
| $L_X$ (derived, $4\pi d^2 F_X$) | $1.8 \times 10^{31}$ erg/s | 7.6% ($\approx 1.4\times10^{30}$ erg/s) | Propagated from $d$ and $F_X$: $\sigma_{L_X}/L_X = \sqrt{(2\sigma_d/d)^2 + \sigma_{F_X}^2}$ |
| $kT_{\rm max}$ (MKCFLOW) | 16.3 keV | 1.15 keV | 90% CI (+1.9/−1.8 keV) $\to$ $1\sigma$ via $\div 1.645$ |

---

### Physical Constants and Auxiliary Quantities

| Symbol | Value | Units | Notes |
|:---|:---|:---|:---|
| $G$ | $6.674 \times 10^{-8}$ | cm$^3$ g$^{-1}$ s$^{-2}$ | Gravitational constant |
| $M_\odot$ | $1.989 \times 10^{33}$ | g | Solar mass |
| $R_\odot$ | $6.957 \times 10^{10}$ | cm | Solar radius |
| $m_H$ | $1.6726 \times 10^{-24}$ | g | Proton mass |
| $\mu_{\rm mol}$ | 0.617 | — | Mean molecular weight, fully ionized solar composition |
| $\alpha_{\rm Yu}$ | 0.646 | — | Empirical shock-temperature correction (Yu+2018; calibrated on 11 nmCVs) |
| $R(M)$, $I(M)$ | Tabulated | cm, g cm$^2$ | Zero-temperature, C-12 WD mass–radius–inertia relation; cubic interpolation. Range: $\sim$0.01–0.99 $M_\odot$ |

---

### Forward Models

Because CJ2056 shows hard thermal X-ray emission ($kT \approx 16$ keV), which requires a standing accretion shock on the WD surface, **we restrict the IP model to the accretion regime** ($r_m \in [R_{\rm WD},\, r_{\rm co}]$). Parameter combinations that place the magnetosphere outside this range (propeller or buried-field) are rejected by the forward models (return NaN $\to$ $\ln\mathcal{L} = -10^{100}$).

**$\dot{P}$ models** (Bhattacharyya & Chakrabarty 2017):
- The spin-evolution equation is $\dot{P}_{\rm pred} = -P^2 \tau / (2\pi I)$, where $\tau$ is the net torque.
- *nmCV:* $\tau = \dot{M}\sqrt{GM R_{\rm WD}}$ — pure spin-up from disk angular momentum deposited at the WD surface.
- *IP (accretion branch, Eq. 9):* $\tau = \tau_{\rm acc} + \tau_{\rm mag}$, where
  - $\tau_{\rm acc} = \dot{M}\sqrt{GM r_m}$ (accretion spin-up)
  - $\tau_{\rm mag} = \frac{\mu^2}{9 r_m^3}\left(2x^3 - 6x^{3/2} + 3\right)$, $x \equiv r_m / r_{\rm co}$ (magnetic threading; can spin up or down depending on $x$)
  - $r_m = \xi \left(\frac{\mu^4}{2GM\dot{M}^2}\right)^{1/7}$ (magnetospheric radius, Ghosh & Lamb 1979)
  - $r_{\rm co} = \left(\frac{GMP^2}{4\pi^2}\right)^{1/3}$ (corotation radius)

**Spectral forward models:**
- *nmCV:* Boundary-layer shock temperature with empirical calibration:
  - $kT_{\rm pred} = \frac{3}{16}\,\alpha_{\rm Yu}\,\mu_{\rm mol}\,m_H\,\frac{GM}{R}$ (Yu+2018)
  - $L_{X,\rm pred} = f_{\rm band}(kT_{\rm pred}) \cdot \frac{GM\dot{M}}{R}$
- *IP:* Rankine-Hugoniot strong-shock maximum ($\gamma = 5/3$), free-fall from $r_m$ to $R_{\rm WD}$:
  - $kT_{\rm pred} = \frac{3}{16}\,\mu_{\rm mol}\,m_H\,v_{\rm ff}^2$, where $v_{\rm ff}^2 = 2GM\left(\frac{1}{R} - \frac{1}{r_m}\right)$
  - $L_{X,\rm pred} = f_{\rm band}(kT_{\rm pred}) \cdot GM\dot{M}\left(\frac{1}{R} - \frac{1}{r_m}\right)$
- The band-correction factor $f_{\rm band}(kT) = e^{-E_{\rm lo}/kT} - e^{-E_{\rm hi}/kT}$ converts bolometric to 0.3–12 keV, assuming single-temperature optically thin bremsstrahlung with Gaunt factor $g_{\rm ff} \approx 1$.

**Key modelling choices and their justifications:**
- *No $\alpha_{\rm Yu}$ correction for the IP:* The Yu+2018 calibration is specific to boundary-layer shocks in nmCVs. IPs form magnetically channeled accretion columns with different shock geometry, so the unadjusted Rankine-Hugoniot result is used.
- *No propeller regime:* The observed hard X-ray emission requires an accretion shock. Propeller systems eject material at $r_m$ and do not produce standing shocks on the WD surface. We therefore exclude the propeller branch a priori.
- *Single-temperature $f_{\rm band}$:* The real post-shock column is multi-temperature (MKCFLOW models a cooling flow), so this introduces a $\sim$20–30% systematic in $L_{X,\rm pred}$. This is a standard approximation; the systematic is absorbed into the $\dot{M}$ posterior.

---

### Prior Specification

We chose **population-based, physics-independent priors** to avoid circular reasoning (the spectral data enter only through the likelihood, not the prior).

| Parameter | Symbol | nmCV prior | IP prior | Justification |
|:---|:---|:---|:---|:---|
| WD mass | $M_{\rm WD}$ ($M_\odot$) | $\mathcal{N}(0.83,\, 0.20)$, clipped to table range | same | Pala+2022 volume-limited CV sample mean and dispersion |
| Log accretion rate | $\log_{10}\dot{M}$ (g/s) | $\mathcal{U}(13,\, 17)$ | same | Broad, uninformative; spans dwarf novae ($\sim10^{13}$) to nova-like ($\sim10^{17}$) |
| Log magnetic moment | $\log_{10}\mu$ (G cm$^3$) | — | $\mathcal{U}(29,\, 35)$ | Covers the full observed IP range ($\sim10^{29}$–$10^{34}$), conservative |
| Magnetospheric prefactor | $\xi$ | — | log-$\mathcal{U}(0.3,\, 1.0)$ | Ghosh & Lamb (1979) disk-accretion range; log-uniform ensures geometric centering of the $\xi^{-7/4}$ systematic on $B$ |

- The Gaussian mass prior is clipped to $[M_{\rm min,table} + 0.001,\; M_{\rm max,table} - 0.001]$ to avoid cubic-interpolation edge effects on $R(M)$ and $I(M)$.
- **Derived quantity:** The surface magnetic field $B = \mu / R^3$ is not a sampled parameter but is computed from the posterior samples of $M_{\rm WD}$ (which determines $R$) and $\mu$.

---

### Likelihood

All three observables are treated as **independent Gaussians**:

$$\ln\mathcal{L} = -\frac{1}{2}\left[\left(\frac{\dot{P}_{\rm obs} - \dot{P}_{\rm pred}}{\sigma_{\dot{P}}}\right)^2 + \left(\frac{kT_{\rm obs} - kT_{\rm pred}}{\sigma_{kT}}\right)^2 + \left(\frac{L_{X,\rm obs} - L_{X,\rm pred}}{\sigma_{L_X}}\right)^2\right]$$

- Samples outside the accretion range ($r_m \notin [R_{\rm WD}, r_{\rm co}]$) or with unphysical predictions ($kT \le 0$, $L_X \le 0$, NaN) receive $\ln\mathcal{L} = -10^{100}$.
- **Independence assumption:** $kT$ and $L_X$ come from the same MKCFLOW spectral fit and share correlated systematics (e.g., $N_H$). We do not propagate this covariance. This likely makes the posterior slightly too narrow but affects both models symmetrically.

---

### Inference Engine

| Setting | Value | Notes |
|:---|:---|:---|
| Algorithm | `dynesty.DynamicNestedSampler` | Dynamic nested sampling; adaptively allocates live points for evidence accuracy |
| $n_{\rm live}$ | 2000 | Conservative for 2D and 4D problems; ensures stable $\ln Z$ |
| Bounding | `'multi'` | Multi-ellipsoid decomposition |
| Sampling | `'rwalk'` | Random-walk proposal; efficient for low-dimensional posteriors |
| Random seed | `np.random.Generator(PCG64(seed))` | Seeds 42–47 for the six sampler runs, for reproducibility |

- **Model evidence** $\ln Z$ and its numerical uncertainty $\sigma_{\ln Z}$ are extracted from `results['logz'][-1]` and `results['logzerr'][-1]`.
- **Equal-weight posterior samples** are obtained via `dynesty.utils.resample_equal`.

---

### Model Comparison

- The **Bayes factor** $\mathcal{B}_{\rm IP/nmCV} = Z_{\rm IP} / Z_{\rm nmCV}$ is computed as $\ln\mathcal{B} = \ln Z_{\rm IP} - \ln Z_{\rm nmCV}$.
- Uncertainty: $\sigma_{\ln\mathcal{B}} = \sqrt{\sigma_{\ln Z_{\rm IP}}^2 + \sigma_{\ln Z_{\rm nmCV}}^2}$.
- Interpretation follows the **Jeffreys (1961) scale**: $|\ln\mathcal{B}| < 1$ = "not worth more than a bare mention"; $1$–$2.5$ = "substantial"; $2.5$–$5$ = "strong"; $> 5$ = "decisive".
- Note: The IP model has 2 extra parameters ($\log_{10}\mu$, $\xi$), so the Bayes factor includes an intrinsic **Occam penalty** proportional to the unused prior volume. With a 6-dex $\mu$ prior, this penalty is non-trivial.

---

### Analysis Steps

1. **Main inference (Cells 9–14):** Run nested sampling for both IP (4D) and nmCV (2D). Extract posteriors, compute Bayes factor.

2. **Corner plots with prior overlays (Cells 16–17):** Visualize the posterior in both a prior-range view (to see how much the data have constrained relative to the prior) and a posterior-zoom view (to resolve the posterior structure). Prior PDFs overlaid on the 1D marginals.

3. **Predicted $\dot{P}$ distribution (Cell 19):** For each posterior sample, compute the model-predicted $\dot{P}$ and compare to the observed $\dot{P} = 0 \pm 1.7\times10^{-15}$. This checks internal consistency: does the spectral-constrained posterior predict a $\dot{P}$ compatible with the timing measurement?

4. **Posterior predictive check (Cell 21):** Compute predicted $kT$ and $L_X$ from the posterior and compare to observations. Reports per-observable $\chi^2$ contributions to verify the models reproduce the input data.

5. **Spin-equilibrium diagnostics (Cells 22–23):**
   - $r_m / r_{\rm co}$ posterior: values near 1 indicate the system is close to spin equilibrium (accretion and magnetic torques nearly cancel).
   - Spin-evolution timescale $\tau = P / |\dot{P}|$: if $\tau \gg$ Gyr, the spin period is effectively frozen — consistent with long-term equilibrium expected for evolved disk-fed IPs.

6. **Sensitivity: old vs. new $\dot{P}$ constraint (Cells 25–28):** Re-run the full inference with the old, weak constraint ($\sigma_{\dot{P}} = 1.8 \times 10^{-12}$ s/s, effectively no timing information). Comparing posteriors and Bayes factors isolates the impact of the improved timing precision.

7. **Sensitivity: spectral-only leverage check (Cells 30–31):** Re-run with $\dot{P}$ removed entirely from the likelihood ($kT + L_X$ only). If the posteriors are unchanged, then $\dot{P}$ is a consistency check, not a constraint — the spectral data already pin the system near spin equilibrium where $\dot{P} \approx 0$.

---

### Caveats

- **$f_{\rm band}$ approximation:** Single-temperature bremsstrahlung; real spectra are multi-temperature cooling flows. Introduces $\sim$20–30% systematic on $L_{X,\rm pred}$.
- **$\alpha_{\rm Yu}$ uncertainty:** Calibrated on only 11 nmCVs (Yu+2018). The nmCV $kT$ prediction and thus the $M_{\rm WD}$ constraint depend directly on this value.
- **Flux uncertainty (5%):** Statistical only; systematic errors from absorption, spectral model choice, and calibration could be 15–30%.
- **$L_X$–$kT$ independence:** Correlated measurement systematics are not propagated.
- **IP prior volume:** The 6-dex $\log_{10}\mu$ prior inflates the Occam penalty against the IP model. A narrower, physically motivated range would reduce this penalty.


## Summary

This analysis fits CJ2056 as an accreting IP (with a non-magnetic CV as a control) using:
1. Broad population priors on $M_{\rm WD}$ and $\dot M$, independent of spectral fits
2. $L_X$ and $kT_{\rm max}$ as observational constraints in the likelihood
3. $\xi$ sampled as a nuisance parameter so that the factor-$\xi^{-7/4}$ systematic on $B$ is absorbed into the posterior
4. Forward-model-level rejection of $r_m$ outside $[R_{\rm WD}, r_{\rm co}]$ — the observed hard thermal X-ray emission is an accretion-shock signature, so the propeller regime is excluded a priori

The physical equations $L_X = GM\dot{M}/R$ and $kT \propto GM/R$ naturally encode the M–$\dot{M}$ anticorrelation; the $L_X$ constraint pins $\dot{M}$ to $\sim 10^{14}$ g/s (consistent with CJ2056 being a low-luminosity IP), which propagates through $r_m \propto (\mu^4/\dot{M}^2)^{1/7}$ to tighten the $\mu$ (and thus $B$) posterior.

### Spin-equilibrium diagnostics

- The IP posterior yields $r_m / r_{\rm co} = 0.70^{+0.19}_{-0.20}$ (68%), well below exact corotation but inside the accretion regime.
- The spin-evolution timescale $\tau = P / |\dot P|$ has median $\sim 5$ Gyr (68% interval 1.8–10 Gyr), exceeding the Hubble time — consistent with long-term spin equilibrium on timescales longer than the age of the Galaxy.

### Interpretation of the spin-equilibrium diagnostics

**Are these diagnostics validity checks, or scientific inferences?** Both — but they serve different roles and carry different levels of model dependence.

*As a validity check (model-independent level).* The tight $\dot P$ measurement $|\dot P|\lesssim 2\times10^{-15}$ s/s forces $\tau = P/|\dot P|\gg$ Gyr *regardless* of any torque model: this is a direct kinematic statement. The fact that the IP posterior can simultaneously satisfy the $kT$, $L_X$, and $\dot P$ data inside the accretion regime (and not be pushed into unphysical corners of parameter space) is a consistency test of the IP framework. If the torque model were grossly wrong, the posterior would either fail to find support or concentrate at the prior edges; neither happens here.

*As a scientific inference (model-conditional level).* Conditional on the Bhattacharyya & Chakrabarty (2017) torque prescription, the posterior location $r_m/r_{\rm co}\sim 0.7$ is the *equilibrium fastness* at which the accretion spin-up torque and the outer-disk magnetic spin-down torque cancel. This yields physical outputs — a few-kG surface field, $\dot M\sim 10^{14}$ g/s, and a multi-Gyr spin-relaxation timescale — that characterise CJ2056 as an evolved, disk-fed low-$B$ IP sitting at its torque equilibrium.

*What model misspecification does (and does not) invalidate.* The *qualitative* conclusion — the accretion and magnetic torques nearly balance, and the system is near its equilibrium fastness — is robust across the common disk-torque prescriptions (Ghosh & Lamb 1979; Wang 1995; Rappaport+2004; B&C 2017), because all of them produce a torque curve that crosses zero at some $r_m/r_{\rm co}<1$ and because the data directly require $|\dot P|/P\ll 1/\tau_{\rm gyr}$. What *is* prescription-dependent is the quantitative numerical value of $r_m/r_{\rm co}$ at equilibrium, and hence the inferred $\mu$ (and $B$) at the $\mathcal{O}(10\text{--}30\%)$ level. So the statement "CJ2056 is at torque equilibrium in a low-$B$ IP geometry" survives model misspecification; the statement "$r_m/r_{\rm co}=0.70$" does not, and the quoted $B$ should be read with a corresponding systematic.

**Why does $r_m/r_{\rm co}$ peak near 0.7 rather than pile up at 1?** Spin equilibrium is *not* the condition $r_m=r_{\rm co}$; it is the condition that the *net* torque vanishes. In the B&C 2017 prescription (and in Ghosh & Lamb 1979 more generally), the spin-up torque from material accreting onto the WD $\tau_{\rm acc}\propto\dot M\sqrt{GM\,r_m}$ is partially compensated by a magnetic spin-down torque that is exerted by field lines threading the disk *outside* $r_m$ — crucially including the region between $r_m$ and $r_{\rm co}$. The net torque therefore crosses zero at a "fastness" $\omega_s=(r_m/r_{\rm co})^{3/2}$ strictly less than unity; for the functional form used here this corresponds to $r_m/r_{\rm co}\sim 0.7\text{--}0.95$ depending on the details of the magnetic coupling. Finding the posterior centred at $\sim 0.7$ with a broad, left-skewed spread is therefore exactly what one expects if the data are driving the system to the equilibrium fastness of this torque law: the posterior is not being pushed to the boundary $r_m=r_{\rm co}$ (which would indicate the sampler hitting the propeller cutoff), but to the *interior* zero of $\dot P(\theta)$. The residual spread reflects (i) the natural $\mu$–$\dot M$ degeneracy in $r_m\propto\mu^{4/7}\dot M^{-2/7}$, which the broad 6-dex $\mu$ prior permits, and (ii) the finite tolerance the $\dot P$ likelihood allows around exact equilibrium. The left-skewed tail toward smaller $r_m/r_{\rm co}$ corresponds to samples with modestly larger $\dot M$ or smaller $\mu$ that still produce $\dot P$ within the observed uncertainty.

### What the precise $\dot P$ measurement adds

The new $\dot P$ constraint ($\sigma = 1.7\times10^{-15}$ s/s) does not measurably tighten the parameter posteriors over the loose old constraint, nor over a spectral-only fit (see the leverage check above).  This is **not a model failure**; it is a physical statement:

- The $kT$ + $L_X$ data already pin $r_m\sim(0.5\text{--}0.9)\,r_{\rm co}$, a region where the accretion and magnetic torques nearly cancel.  CJ2056 is therefore consistent with **long-term spin equilibrium** ($P/|\dot P|\sim$ few Gyr), the regime expected for evolved disk-fed IPs.
- The $\dot P$ measurement is therefore best read as an **independent confirmation** of the spectral picture.  Had $|\dot P|$ been measured at $\sim 10^{-13}$ s/s (well above the equilibrium prediction), the spectral fit would be falsified.  The non-detection rules that out.
- Under the IP interpretation we obtain a magnetic-field constraint $B\sim$ a few kG (with $\sim$factor-3 systematic from $\xi$ and an additional $\mathcal{O}(10\text{--}30\%)$ systematic from the choice of torque prescription) — useful for population-level studies of low-$B$ IPs.
- The Bayes factor weakly favours nmCV over IP (ln BF $\sim -3.5$), with the conclusion robust to the $\dot P$ precision.  In other words, the data do not decisively classify CJ2056, and the precise timing measurement, while valuable as a falsifiability check, is not the discriminating observable for this system.